In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import test_transforms
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir)
transformed_dataset = ImageDataset(annot_path, img_dir, test_transforms)

In [3]:
_, truth_labels = next(iter(transformed_dataset))
truth_labels.shape, truth_labels

(torch.Size([5, 5]),
 tensor([[8.0000, 0.5848, 0.7321, 0.1205, 0.3393],
         [8.0000, 0.4196, 0.8482, 0.1741, 0.2902],
         [8.0000, 0.0714, 0.8259, 0.1250, 0.3482],
         [8.0000, 0.5357, 0.6562, 0.1071, 0.2812],
         [8.0000, 0.5893, 0.5402, 0.0714, 0.0893]]))

In [4]:
bboxes = truth_labels[:, -4:]
bboxes.shape, bboxes

(torch.Size([5, 4]),
 tensor([[0.5848, 0.7321, 0.1205, 0.3393],
         [0.4196, 0.8482, 0.1741, 0.2902],
         [0.0714, 0.8259, 0.1250, 0.3482],
         [0.5357, 0.6562, 0.1071, 0.2812],
         [0.5893, 0.5402, 0.0714, 0.0893]]))

In [5]:
truth_labels[:, 0]

tensor([8., 8., 8., 8., 8.])

In [6]:
bbox_preds = torch.randn(100, 21)
bbox_preds

tensor([[ 0.4559, -0.1178, -0.2685,  ..., -0.4621,  0.1172,  0.5419],
        [ 0.4290, -0.8091,  1.8188,  ...,  0.2553, -0.9118,  0.6817],
        [ 1.0498, -1.6466,  0.5720,  ...,  0.6131,  0.1883, -0.1951],
        ...,
        [-0.0817, -0.2526, -0.2476,  ..., -0.2318,  1.1715, -1.5956],
        [ 0.3145,  0.1381,  1.0878,  ..., -0.6004, -0.1176, -0.2567],
        [ 2.3444,  1.0234, -1.4074,  ...,  0.9582,  1.6382,  0.2747]])

In [7]:
class_preds = torch.randn(100, 21)

In [8]:
for idx in truth_labels[:, 0]: print(int(idx))

8
8
8
8
8


In [9]:
import torch.nn.functional as F

In [10]:
from src.utilities import giou

bbox_preds = torch.randn(100, 4)

In [11]:
truth_boxes = truth_labels[..., -4:]
truth_boxes.shape

torch.Size([5, 4])

In [12]:
 truth_boxes[0], bbox_preds[0], bbox_preds[1], bbox_preds[2], bbox_preds[3], bbox_preds[4]

(tensor([0.5848, 0.7321, 0.1205, 0.3393]),
 tensor([-1.7803, -0.6810,  0.8188, -1.4222]),
 tensor([-0.2097, -0.8453,  0.2236, -1.2325]),
 tensor([-0.1551, -1.1009,  0.3324,  1.3840]),
 tensor([ 3.5675e-01, -9.1648e-04,  6.2784e-01, -9.2499e-01]),
 tensor([ 0.9212, -1.4138,  0.2749,  0.5401]))

In [13]:
preds_clone = bbox_preds.clone().detach()

# INTERSECTION COORDINATES
preds_clone[..., 0] = torch.max(bbox_preds[..., 0], truth_boxes[0][0])
preds_clone[..., 1] = torch.max(bbox_preds[..., 1], truth_boxes[0][1])
preds_clone[..., 2] = torch.min(bbox_preds[..., 2], truth_boxes[0][2])
preds_clone[..., 3] = torch.min(bbox_preds[..., 3], truth_boxes[0][3])

preds_clone.shape, preds_clone[0], preds_clone[1], preds_clone[2], preds_clone[3], preds_clone[4]

(torch.Size([100, 4]),
 tensor([ 0.5848,  0.7321,  0.1205, -1.4222]),
 tensor([ 0.5848,  0.7321,  0.1205, -1.2325]),
 tensor([0.5848, 0.7321, 0.1205, 0.3393]),
 tensor([ 0.5848,  0.7321,  0.1205, -0.9250]),
 tensor([0.9212, 0.7321, 0.1205, 0.3393]))

In [14]:
w = torch.clamp(preds_clone[..., 2] - preds_clone[..., 0], min=0)
h = torch.clamp(preds_clone[..., 3] - preds_clone[..., 1], min=0)

In [15]:
w[0], w[1], h[0], h[1], preds_clone[0], preds_clone[1]

(tensor(0.),
 tensor(0.),
 tensor(0.),
 tensor(0.),
 tensor([ 0.5848,  0.7321,  0.1205, -1.4222]),
 tensor([ 0.5848,  0.7321,  0.1205, -1.2325]))

In [16]:
areas = w * h
areas.shape, areas[0], areas[1]

(torch.Size([100]), tensor(0.), tensor(0.))